# Tu copia personal · Aula UTP

Este cuaderno se abre desde GitHub. En Colab elige **Archivo → Guardar una copia en Drive** antes de conservar cambios. Renombra esa copia y ejecuta de arriba abajo. Cada estudiante trabaja en su propio archivo y entorno.

Los ejemplos son didácticos: sigue la sección indicada por Cristian para esta clase. Las comprobaciones son voluntarias, sin entregas ni notas.

# Laboratorio complementario explicado · Machine Learning UTP
Cristian Alvarado. Datos simulados incluidos. Revisión guiada, sin ejercicios para entregar.

Recorrido: preparación → evaluación → ensambles → agrupación por densidad → pronóstico temporal. Ejecuta las celdas en orden.

## Ruta de esta práctica · ML · Encuentros 1–4
1: preparación/regresión; 2: conteos y métricas; 3: ensambles; 4: DBSCAN; 5: series temporales. Los ejemplos de refuerzo y PCA están en cuadernos propios.

**Antes de comenzar:** guarda tu copia personal. Usa el índice lateral de Colab para localizar el bloque indicado. Ejecuta las celdas de preparación y después el bloque, en orden; cambiar una variable exige volver a ejecutar su celda y las que dependen de ella. Abrir una figura no vuelve a entrenar un modelo.

**Entorno:** estos ejemplos están preparados para CPU. No requieren claves de API, contratar servicios ni activar un entorno de pago. La disponibilidad y límites de los recursos de Colab dependen de tu cuenta; no cambies a recursos de pago para esta práctica. Si se interrumpe la sesión, conserva el cuaderno y vuelve a ejecutar desde la preparación.

**Distingue siempre:** datos observados o simulados → parámetros aprendidos por `fit`/entrenamiento → configuraciones que tú eliges → resultados en datos reservados. Una buena cifra de entrenamiento no demuestra aprendizaje útil en nuevos casos.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.cluster import DBSCAN
rng = np.random.default_rng(42)


## 1. ¿Podemos estimar el rendimiento de una finca?

**Pregunta:** con lluvia, temperatura y fertilizante, ¿qué rendimiento estimaría un modelo para una finca que no utilizó al aprender?

Cada fila representa una finca en una temporada. Las entradas **X** son lluvia en **mm**, temperatura en **°C** y fertilizante en **kg/ha**. La respuesta **y** es rendimiento en **t/ha**: toneladas por hectárea.

Usamos **240 fincas simuladas**, no mediciones de campo. Las medidas corresponden a un periodo definido antes de la cosecha y el rendimiento al final de la temporada. Esta simplificación sirve para recorrer el proceso; no es una recomendación de manejo agrícola.

Ejecuta primero la celda de importaciones de arriba. `rng` es el generador de números aleatorios; la semilla 42 permite reproducir el ejemplo al ejecutar el cuaderno desde el comienzo.

### 1.1. Crear la tabla y su respuesta conocida

Generamos lluvia entre 200 y 800 mm, temperatura entre 16 y 30 °C y fertilizante entre 50 y 180 kg/ha. Son rangos elegidos para la simulación, no límites agronómicos generales.

El rendimiento **también se inventa** con una regla: aumenta con lluvia y fertilizante; disminuye al alejarse de 23 °C; lleva una pequeña variación aleatoria. Los números de esa regla son decisiones didácticas. Así conocemos una respuesta para cada fila y podemos comprobar después las predicciones.

`rng.uniform` genera las entradas dentro de los rangos. `rng.normal` añade variación. `**2` eleva al cuadrado la diferencia de temperatura: crea una relación curva. El modelo de regresión aprenderá de las filas; no recibe esta regla generadora.

**Lee la salida:** cada fila muestra las tres entradas y el rendimiento simulado correspondiente.

In [ ]:
n = 240
X = pd.DataFrame({
    'lluvia_mm': rng.uniform(200, 800, n),
    'temperatura_c': rng.uniform(16, 30, n),
    'fertilizante_kg_ha': rng.uniform(50, 180, n)
})
y = (1 + .004 * X.lluvia_mm + .015 * X.fertilizante_kg_ha
     - .03 * (X.temperatura_c - 23)**2 + rng.normal(0, .25, n))
display(X.assign(rendimiento_t_ha=y).head())


### 1.2. Simular registros de lluvia que faltan

Ocultamos la lluvia de **20 fincas**, elegidas al azar sin repetir. `NaN` significa dato ausente; no significa cero lluvia. Mantenemos su rendimiento conocido: pudo registrarse la cosecha aunque faltara la medición de lluvia.

La respuesta se creó **antes** de ocultar los datos para poder evaluar este caso didáctico. Más adelante completaremos las entradas ausentes con una estimación llamada **imputación**.

**Lee la salida:** debe haber 20 valores de lluvia sin registro. Las filas mostradas ayudan a reconocer cómo aparece un dato ausente en una tabla.

In [ ]:
X.loc[rng.choice(n, 20, replace=False), 'lluvia_mm'] = np.nan
print('Fincas sin registro de lluvia:', X.lluvia_mm.isna().sum())
display(X.loc[X.lluvia_mm.isna()].head(3))


### 1.3. Separar antes de aprender

Reservamos **180 fincas para entrenamiento** y **60 para prueba**. `test_size=.25` aparta el 25 %; `random_state=42` reproduce esa partición.

`X_tr` y `y_tr` son entradas y respuestas de entrenamiento. `X_te` y `y_te` son las de prueba. El modelo aprende con las primeras; sus predicciones se comprueban con las respuestas reservadas.

La mediana para completar vacíos y la escala también se aprenden solo con entrenamiento. Usar prueba para calcularlas introduciría información que debía permanecer reservada: **fuga de información**.

Aquí tratamos las fincas como independientes. Para mediciones repetidas de una finca o pronósticos por fecha, habría que separar por finca o respetar el tiempo. En este ejemplo no elegimos hiperparámetros; si comparáramos muchas configuraciones, necesitaríamos validación dentro de entrenamiento.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=.25, random_state=42
)
print('Fincas de entrenamiento:', len(X_tr))
print('Fincas de prueba:', len(X_te))


### 1.4. Crear una referencia sencilla

Antes de usar una regresión, necesitamos una respuesta básica contra la cual comparar. `DummyRegressor(strategy='mean')` aprende la **media del rendimiento de entrenamiento** y predice ese mismo valor para todas las fincas de prueba. No utiliza lluvia, temperatura ni fertilizante para distinguirlas.

`make_pipeline` encadena pasos. En esta referencia, `SimpleImputer(strategy='median')` completa entradas ausentes y después actúa el predictor de media. La imputación mantiene una preparación compatible, aunque este predictor no depende de las entradas.

`fit` aprende esos pasos únicamente con entrenamiento. **Lee la salida:** el número es la predicción de referencia, en t/ha; no es su error.

In [ ]:
base = make_pipeline(
    SimpleImputer(strategy='median'),
    DummyRegressor(strategy='mean')
)
base.fit(X_tr, y_tr)
print(f'La referencia predice {base.predict(X_te)[0]:.3f} t/ha para cada finca.')


### 1.5. Aprender una regresión con las tres entradas

Este segundo pipeline hace tres pasos en orden:

1. **Imputar:** completa una lluvia ausente con la mediana aprendida de las lluvias de entrenamiento.
2. **Estandarizar:** expresa cada entrada respecto a su media y su dispersión, aprendidas con entrenamiento. Así se manejan magnitudes diferentes; esto no añade información ni garantiza una mejora.
3. **Regresión lineal:** aprende un valor inicial y un peso para cada entrada para estimar rendimiento.

Para datos de prueba, el pipeline aplica las mismas reglas y los pesos ya aprendidos; no los recalcula. La relación de temperatura que inventamos es curva y esta regresión usa las tres variables sin potencias: puede quedar error por esa simplificación y por la variación aleatoria.

**Lee la salida:** la mediana está en mm y es la que usará para cualquier lluvia ausente. No tiene por qué ser 600 mm: ese número pertenecía al ejemplo ilustrativo de cuatro fincas de la presentación.

In [ ]:
lineal = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    LinearRegression()
)
lineal.fit(X_tr, y_tr)
print(f'Mediana de lluvia aprendida: {lineal[0].statistics_[0]:.3f} mm')


### 1.6. Comparar predicciones con respuestas reservadas

`predict(X_te)` estima rendimientos para las 60 fincas de prueba. Después comparamos cada predicción con su rendimiento conocido, `y_te`. Llamamos **error absoluto** a la distancia entre ambos valores, sin signo; el **MAE** es el promedio de esas distancias: `MAE = promedio(|y − ŷ|)`, donde `ŷ` es la predicción.

**Cómo leer la tabla y el gráfico:**

- Ambos modelos se evalúan sobre las mismas 60 fincas. Cada barra muestra su MAE, no el rendimiento promedio.
- El eje vertical usa t/ha. Un MAE de 0,46 significa un error absoluto promedio cercano a 0,46 toneladas por hectárea en esta prueba; no que todas las fincas tengan ese error.
- La barra más baja corresponde al menor error medio. Compara la regresión con la referencia para saber si las entradas ayudaron en este ejemplo.
- La última tabla muestra cinco fincas de prueba: respuesta conocida, predicción de regresión y error absoluto de cada una.

**Alcance:** con una ejecución completa desde la semilla 42, la referencia obtiene aproximadamente **0,705 t/ha** y la regresión **0,459 t/ha**. La regresión mejora esta referencia en esta simulación. No demuestra que funcionará igual en fincas reales, ni que una variable cause cambios en el rendimiento. Las cifras pueden cambiar si alteras los datos o ejecutas celdas aleatorias de nuevo sin reiniciar el recorrido.

In [ ]:
pred_base = base.predict(X_te)
pred_lineal = lineal.predict(X_te)
resultados = pd.DataFrame({
    'modelo': ['Media de entrenamiento', 'Regresión lineal'],
    'MAE_t_ha': [mean_absolute_error(y_te, pred_base),
                 mean_absolute_error(y_te, pred_lineal)]
})
display(resultados)
plt.bar(resultados.modelo, resultados.MAE_t_ha)
plt.ylabel('MAE en prueba (t/ha)')
plt.title('Comparar con una referencia: mismas 60 fincas de prueba')
plt.show()
comparacion = pd.DataFrame({
    'rendimiento_conocido_t_ha': y_te,
    'prediccion_lineal_t_ha': pred_lineal,
    'error_absoluto_t_ha': np.abs(y_te - pred_lineal)
})
display(comparacion.head().round(3))


### Lectura del gráfico 1 · ¿Las entradas mejoraron la referencia?

**Eje horizontal:** los dos modelos. **Eje vertical:** error absoluto medio en las 60 fincas de prueba, expresado en t/ha. La altura de una barra representa error, no producción ni número de fincas. Cada barra resume 60 comparaciones entre respuesta conocida y predicción.

En esta ejecución, la referencia tiene **0,705 t/ha** de MAE y la regresión **0,459 t/ha**. La segunda barra es más baja: la regresión comete menos error medio sobre la misma prueba. Eso es lo que podemos concluir para este ejemplo.

En la última tabla, una finca tiene rendimiento conocido **4,338**, predicción **4,889** y error absoluto **0,551 t/ha**. La predicción quedó por encima de la respuesta; otra fila puede quedar por debajo. El MAE promedia las distancias sin compensar unos errores con otros. No es un porcentaje de aciertos ni el error de todas las fincas.

**Conexión con la presentación:** la diapositiva 6 resume aprender y predecir; aquí ves las filas, los pasos y el error que produce ese recorrido. La diapositiva 8 usa cuatro fincas ilustrativas para explicar la mediana; este laboratorio calcula su propia mediana con sus 180 fincas de entrenamiento.

Documentación de consulta: [imputación](https://scikit-learn.org/stable/modules/impute.html), [pipelines](https://scikit-learn.org/stable/modules/compose.html#pipeline), [MAE](https://scikit-learn.org/stable/modules/model_evaluation.html#mean-absolute-error).

## 2. Contar errores de clasificación
En este ejemplo de 10 plantas, positivo significa con plaga. Filas = realidad, columnas = predicción; el orden es sin plaga, con plaga. Cuenta TP=3, FN=1, FP=2 y TN=4. Mira por qué precision y recall responden preguntas distintas.

In [ ]:
real=np.array([1,1,1,1,0,0,0,0,0,0])
pred=np.array([1,1,1,0,1,1,0,0,0,0])
cm=confusion_matrix(real,pred,labels=[0,1])
display(pd.DataFrame(cm,index=['Real sin plaga','Real con plaga'],columns=['Pred sin plaga','Pred con plaga']))
display(pd.DataFrame({'métrica':['Accuracy','Precision','Recall','F1'],'valor':[accuracy_score(real,pred),precision_score(real,pred),recall_score(real,pred),f1_score(real,pred)]}))
fig,ax=plt.subplots();ax.imshow(cm,cmap='Blues')
for i in range(2):
    for j in range(2): ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=20)
ax.set_xticks([0,1],['Sin plaga','Con plaga']);ax.set_yticks([0,1],['Sin plaga','Con plaga']);ax.set_xlabel('Predicción');ax.set_ylabel('Realidad');ax.set_title('Matriz de confusión · 10 plantas');plt.show()


### Lectura del gráfico 2 · ¿Qué errores cometimos en diez plantas?

Esta imagen es una **matriz de conteos**, no una curva. El eje vertical indica la **realidad** y el horizontal la **predicción**. Para leer una casilla, combina su fila y su columna. El azul más oscuro significa más plantas en esa casilla, no mayor calidad.

| Realidad → predicción | Plantas | Interpretación |
|---|---:|---|
| Sin plaga → sin plaga | 4 | Sanas correctamente descartadas (TN). |
| Sin plaga → con plaga | 2 | Falsas alertas (FP). |
| Con plaga → sin plaga | 1 | Plaga que se pasó por alto (FN). |
| Con plaga → con plaga | 3 | Plagas detectadas (TP). |

La diagonal reúne los **7 aciertos**. Fuera de ella están **3 errores**, con consecuencias distintas: revisar una planta sana o dejar pasar una planta con plaga.

La tabla de métricas usa decimales: **accuracy 0,70 = 70 %** de aciertos; **precision 0,60 = 60 %** de las alertas correctas; **recall 0,75 = 75 %** de las plantas con plaga detectadas. **F1 ≈ 0,667** es su media armónica: `F1 = 2 × precision × recall / (precision + recall)`. Ninguna métrica por sí sola dice cuánto cuesta cada error. Estas diez respuestas son un ejemplo construido; aquí no entrenamos un clasificador.

### Encuentros 3 y 4: nombres completos de las medidas y métodos

Estas palabras ya aparecen en esta práctica. El nombre completo ayuda a leer el código; no añade una tarea nueva.

- **MAE** — MAE: Mean Absolute Error (Error absoluto medio). Promedio de las diferencias sin signo entre cada predicción y su respuesta observada: se resta, se elimina el signo y se promedian las distancias. Menor es mejor sobre los mismos casos y unidades; 0 indica coincidencia en todos esos casos. Conserva la unidad de la respuesta y no significa que cada caso tenga ese mismo error. Si se observan 4 y 10 cajas y se predicen 3 y 7, los errores absolutos son 1 y 3 cajas: MAE = (1+3)/2 = 2 cajas. [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_error.html)

- **MSE** — MSE: Mean Squared Error (Error cuadrático medio). Promedio de las diferencias al cuadrado entre predicción y respuesta observada: cada diferencia se multiplica por sí misma antes de promediar. Menor es mejor y 0 indica coincidencia; sus unidades están al cuadrado. Los errores grandes pesan más que en el MAE, por lo que ambas métricas pueden preferir modelos diferentes. Con errores de 1 y 3 cajas: MSE = (1²+3²)/2 = (1+9)/2 = 5 cajas². [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html)

- **RMSE** — RMSE: Root Mean Squared Error (Raíz del error cuadrático medio). Raíz cuadrada del promedio de diferencias al cuadrado entre predicción y respuesta observada. Vuelve a las unidades de la respuesta, aunque conserva la mayor influencia de errores grandes del MSE. Menor es mejor sobre los mismos casos; no es el promedio de las distancias absolutas. Con errores de 1 y 3 cajas, MSE = 5 cajas² y RMSE = √5 ≈ 2,24 cajas. [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html)

- **R²** — R²: Coeficiente de determinación. Símbolo de una medida de regresión: R² = 1 − suma de errores cuadrados / suma de diferencias cuadradas respecto a la media observada. Para respuestas no constantes, 1 es ajuste perfecto y 0 rinde lo mismo que predecir su media; puede ser negativo. No es porcentaje de aciertos; con respuestas constantes el cociente no está definido y algunas bibliotecas sustituyen el resultado. [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html)

- **DBSCAN** — DBSCAN: Density-Based Spatial Clustering of Applications with Noise (Agrupación espacial basada en densidad con casos no agrupados). Método que forma grupos conectando zonas con suficientes vecinos. eps fija la distancia máxima entre vecinos y min_samples el mínimo para que un punto sea núcleo, contando al propio punto. No se elige de antemano el número de grupos; −1 identifica puntos sin grupo con esa configuración. El radio no limita la distancia entre todos los miembros de un grupo y quedar aparte no demuestra mala calidad. [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html)


## 3. Comparar ensambles sobre la misma prueba · ML · Encuentro 3

**Dónde estamos en el aula:** este encuentro estudia agrupación, PCA y ensambles. Los números 3, 4 y 5 de este archivo son **bloques del cuaderno**, no números de encuentro. Para seguir el orden del aula, estudia DBSCAN en bloque 4, PCA en su cuaderno propio y vuelve a este bloque 3. K-Means y Mean-Shift son ampliaciones en Agrupación de fincas simuladas. En el encuentro 4 usarás bloque 5 (series), Refuerzo en su cuaderno y una comparación ya ejecutada para integrar.

**Antes de ejecutar este bloque:** recupera la preparación del bloque 1 (importaciones y 1.1–1.6). Eso crea las mismas 180 fincas de entrenamiento, 60 de prueba y la referencia llamada `base`. No tienes que cambiar ese contenido. Si aparece NameError de `X_tr` o `base`, falta ejecutar esas celdas en esta sesión; no se arregla inventando otra tabla.

**Problema:** una cooperativa quiere saber qué estimador de rendimiento merece más pruebas para planear transporte. Una referencia sencilla predice siempre la media aprendida. Un árbol hace preguntas encadenadas sobre las medidas para estimar una respuesta. Random Forest es un bosque que combina varios árboles aprendidos con sorteos de ejemplos y medidas. Gradient boosting añade modelos por etapas para intentar reducir el desacuerdo que deja la combinación anterior.

**La comparación justa:** todos aprenden con las mismas 180 fincas y se comprueban con las mismas 60 reservadas. No son los dos modelos A/B ni el promedio perfecto del pequeño ejemplo visual: sus datos y algoritmos son diferentes. Compararemos error medio en t/ha, no beneficios económicos.

In [ ]:
modelos={'Referencia':base,'Árbol':make_pipeline(SimpleImputer(strategy='median'),DecisionTreeRegressor(max_depth=5,random_state=42)),'Random Forest':make_pipeline(SimpleImputer(strategy='median'),RandomForestRegressor(n_estimators=100,max_depth=6,random_state=42)),'Gradient boosting':make_pipeline(SimpleImputer(strategy='median'),GradientBoostingRegressor(random_state=42))}
filas=[]
for nombre,modelo in modelos.items():
    modelo.fit(X_tr,y_tr);p=modelo.predict(X_te)
    filas.append({'modelo':nombre,'MAE_t_ha':mean_absolute_error(y_te,p),'RMSE_t_ha':np.sqrt(mean_squared_error(y_te,p)),'R2':r2_score(y_te,p)})
tabla=pd.DataFrame(filas);display(tabla)
plt.barh(tabla.modelo,tabla.MAE_t_ha);plt.xlabel('MAE en prueba (t/ha)');plt.title('Mismos datos de prueba para todos');plt.show()


### Lectura del gráfico 3 · ¿Qué estimador se acercó más?

Cada barra horizontal corresponde a un modelo; su longitud es **error**, no cosecha. El eje horizontal usa toneladas por hectárea (t/ha). Más corta significa menor error medio en las mismas 60 fincas de prueba. El orden vertical no clasifica la calidad.

**MAE, antes de su sigla:** para cada finca toma la diferencia entre lo observado y lo predicho sin signo; después promedia esas diferencias. Si los errores fueran 1 y 3 t/ha, su MAE sería (1+3)/2 = 2 t/ha. Son dos números de explicación, no el resultado de estas 60 fincas.

Con la ejecución completa de referencia se obtuvieron aproximadamente 0,705 para referencia, 0,437 para árbol, 0,358 para Random Forest y 0,301 t/ha para boosting. **Lee la tabla de tu ejecución**: si cambiaste datos, configuraciones o versiones, no copies esos números como si fueran salidas nuevas.

La tabla también muestra **RMSE**: eleva cada error al cuadrado, promedia y toma la raíz; conserva t/ha y hace pesar más los errores grandes. **R²** compara el error cuadrático con predecir la media de las respuestas de prueba; puede ser negativo y no es porcentaje de aciertos. El gráfico representa solo MAE: no intentes leer RMSE o R² en la longitud de sus barras.

**Tip para no cambiar dos cosas a la vez:** guarda la tabla actual. Cambia únicamente la profundidad máxima del árbol de 5 a 3 y vuelve a ejecutar este bloque, conservando X_tr/X_te. Cambian el árbol y sus errores; los casos de prueba son los mismos. No tiene que mejorar. La profundidad limita cuántas preguntas encadena y es una configuración elegida, no un valor aprendido.

Elegir repetidamente configuraciones por esta prueba acabaría usándola para desarrollar el modelo. En una aplicación se necesitaría validación durante el desarrollo y una prueba final separada. Este bloque muestra la comparación, no autoriza una selección de producción.

## 4. DBSCAN: organizar una revisión por cercanía · ML · Encuentro 3

**El puente con el aula:** allí cada punto representaba un centro de acopio descrito por dos porcentajes. Aquí usamos 123 puntos con dos números inventados y sin unidad física: 60 alrededor de una zona, 60 de otra y tres apartados. Es una ampliación para ver densidad; no asignamos a estos números el significado de lluvia, calidad o registros reales de centros.

**DBSCAN** agrupa puntos conectados por vecindarios con suficientes casos. **Vecindario** son los puntos dentro de la separación máxima elegida. **eps=0,35** fija esa separación después de estandarizar. **min_samples=5** exige al menos cinco puntos en ese vecindario contando al propio punto. Un punto que no se une a un grupo recibe etiqueta −1; eso no demuestra un error del sensor.

**Preparación:** ejecuta la celda de importaciones y después este bloque. `rng_grupos` es un generador local con semilla 42: cada ejecución de este bloque reproduce los mismos puntos. Así puedes cambiar solo el radio sin cambiar también las observaciones. Este bloque no necesita que hayas ejecutado los modelos de regresión.

Antes del código, predice qué verás: dos concentraciones y algunos puntos que quizá queden aparte. No supongas que todos los 120 puntos centrales deben agruparse ni que solo los tres añadidos serán −1.

In [ ]:
# Generador local: repetir este bloque conserva los mismos puntos al cambiar eps.
rng_grupos = np.random.default_rng(42)
a=rng_grupos.normal([0,0],.25,(60,2));b=rng_grupos.normal([2,2],.25,(60,2));aislados=np.array([[-2,2],[4,-1],[0,4]])
datos=np.vstack([a,b,aislados]);escalados=StandardScaler().fit_transform(datos)
etiquetas=DBSCAN(eps=.35,min_samples=5).fit_predict(escalados)
display(pd.Series(etiquetas).value_counts().sort_index().rename('número de puntos'))
for grupo in np.unique(etiquetas):
    sel=etiquetas==grupo;plt.scatter(datos[sel,0],datos[sel,1],label='No agrupado' if grupo==-1 else f'Grupo {grupo}')
plt.xlabel('Medida 1 (simulada)');plt.ylabel('Medida 2 (simulada)');plt.title('DBSCAN: agrupación por densidad');plt.legend();plt.show()


### Lectura del gráfico 4 · Del color a una decisión de revisión

Cada punto es una observación con dos números inventados: medida 1 en horizontal y medida 2 en vertical. El color se lee con la **leyenda**: Grupo 0 y Grupo 1 son nombres, mientras «No agrupado» corresponde a −1. Ninguno significa bueno o malo.

La tabla de conteos tiene una fila por etiqueta: el número de la etiqueta y cuántos puntos recibió. La suma debe ser **123**. Comprueba los conteos que imprime tu ejecución. La semilla local conserva estos puntos al repetir el bloque; cambiarla o modificar cómo se generan sí crearía otra simulación.

**Una separación importante:** se estandarizaron las medidas antes de agrupar, para compararlas según su propia dispersión. El dibujo muestra las coordenadas originales. Por eso el radio 0,35 no se puede medir directamente con una regla sobre estos ejes. Cambiar la escala también cambia lo que significa parecerse.

**Prueba pequeña manteniendo los puntos:** guarda los conteos actuales. En tu copia cambia únicamente `eps=.35` a `eps=.50` y vuelve a ejecutar este bloque. El generador local reproduce los mismos `datos`: solo cambió la cercanía permitida. Compara las nuevas etiquetas y conteos con los anteriores. El número de grupos puede crecer, reducirse o fusionarse al cambiar configuraciones; lo que debes documentar es qué casos se revisan juntos y cuáles aparte.

Antes de dar una acción a un grupo, vuelve a las medidas originales. En este conjunto no hay significado operativo real que permita diagnosticar una falla: sirve para practicar agrupación, no para decidir manejo agrícola.

## 5. Series temporales: planear con lo conocido · ML · Encuentro 4

**Situación:** una comercializadora quiere anticipar cuánto venderá después del cierre de su historial. Se inventan 84 semanas de ventas; no son registros de un negocio. Aquí «unidad vendida» es una unidad de la simulación, sin presentarla como kg, dinero o cajas de una empresa. El caso pequeño del aula sí usa cajas, pero sus cifras no se sustituyen por las de este conjunto.

La **serie temporal** es la lista ordenada de ventas semanales. El **corte** separa las primeras 60 conocidas de las últimas 24 reservadas. El **horizonte** son esas 24 semanas futuras. El programa añade crecimiento, un ciclo de 12 semanas y variación aleatoria: son elecciones del ejercicio, no un calendario agrícola o una estacionalidad anual demostrada.

Compara tres reglas: repetir la última venta, repetir el último ciclo conocido de 12 semanas o prolongar una recta ajustada con el pasado. El **pronóstico** estima lo que no se sabía al cerrar entrenamiento. Las ventas posteriores aparecen solo para comprobar esas estimaciones.

**Antes de ejecutar:** ejecuta la celda de importaciones del comienzo. Este bloque usa `rng_serie`, un generador local con semilla 42; reproduce la misma serie aunque antes hayas estudiado otros bloques. Para comparar reglas conserva la serie y el mismo corte. Son 24 pronósticos emitidos de una vez: no se actualizan con cada venta futura.

In [ ]:
# Generador local: las reglas siempre se comparan sobre la misma serie simulada.
rng_serie = np.random.default_rng(42)
t=np.arange(84)
ventas=100+.6*t+12*np.sin(2*np.pi*t/12)+rng_serie.normal(0,2,len(t))
corte=60;periodo=12
pasado=ventas[:corte];futuro=ventas[corte:]
t_tr=t[:corte,None];t_te=t[corte:,None]
modelo=LinearRegression().fit(t_tr,pasado)
pronosticos={'Último valor':np.repeat(pasado[-1],len(futuro)),'Referencia estacional':np.tile(pasado[-periodo:],2),'Tendencia lineal':modelo.predict(t_te)}
display(pd.DataFrame({'método':list(pronosticos),'MAE_unidades':[mean_absolute_error(futuro,p) for p in pronosticos.values()]}))
plt.figure(figsize=(10,4));plt.plot(t,ventas,label='Ventas observadas',color='black')
for nombre,p in pronosticos.items(): plt.plot(t[corte:],p,label=nombre)
plt.axvline(corte-.5,linestyle='--',color='gray',label='Fin de entrenamiento');plt.xlabel('Semana');plt.ylabel('Unidades vendidas');plt.title('Pasado para entrenar, futuro para evaluar');plt.legend();plt.show()


### Lectura del gráfico 5 · Primero localiza la fecha de decisión

El eje horizontal numera semanas de **0 a 83**: contar desde cero no elimina una semana. La separación discontinua está entre los índices 59 y 60, después de las 60 semanas conocidas. El eje vertical mide unidades vendidas simuladas. La línea negra es lo observado; a la derecha del corte se enseña después para evaluar, no para ajustar.

**Mira una regla cada vez con la leyenda:**

- Último valor: repite la última venta conocida y dibuja una horizontal.
- Referencia estacional: repite dos veces las últimas 12 semanas conocidas. «Estacional» significa aquí ciclo elegido de 12 semanas, no año agrícola.
- Tendencia lineal: prolonga una recta. Puede captar crecimiento, pero no la oscilación del ciclo.

En una semana reservada, la distancia vertical entre una predicción y la línea negra es su error. El **MAE** promedia las 24 distancias sin signo. Lee los valores que imprime la tabla de esta ejecución; corresponden a esta serie y sus 24 semanas reservadas. La recta puede tener menor error aunque visualmente no siga cada oscilación.

**Tip comprobable:** revisa `len(pasado)` y `len(futuro)`: deben ser 60 y 24. Para ver una semana concreta sin cambiar la simulación, compara `futuro[0]` con cada `p[0]` de los pronósticos. Esa es la primera semana reservada, índice 60. Su diferencia no es el MAE total: este promedia las 24.

Si quieres estudiar otro corte, conserva una copia del resultado y adapta también las referencias al nuevo horizonte; no cambies solo 60 en una línea y des por válido que repetir dos ciclos cubre cualquier longitud. Este ejemplo está preparado para 24 semanas y dos ciclos de 12.

## Cierre de ML · Encuentro 4: une una pregunta con su evidencia

No hace falta inventar otro modelo. Vuelve a una comparación ya calculada en el bloque 1 o 3 y explica con tus propias cifras:

1. **Quién decide y cuándo:** por ejemplo, planear transporte antes de conocer la cosecha.
2. **Qué entra y qué sale:** lluvia, temperatura y fertilizante disponibles → rendimiento estimado en t/ha. Multiplicar por hectáreas daría producción total solo si esa superficie se declara.
3. **Cómo se comprobó:** referencia y candidata sobre las mismas fincas reservadas; error con unidad y ejemplos de predicciones junto a respuestas conocidas.
4. **Qué no se ha demostrado:** otras temporadas o condiciones, beneficios económicos, causalidad o recomendaciones de manejo.

**Tip:** puedes describir un solo caso de prueba antes del promedio. Usa la misma fila de `X_te`, su predicción y la respuesta correspondiente de `y_te`; no mezcles el índice de una finca con el de otra. Esa explicación conecta dato → cálculo → error → límite, en lugar de mostrar una tabla sin propósito.

Las comprobaciones son voluntarias, sin entregas ni calificaciones. Fuentes de consulta: [scikit-learn: errores frecuentes](https://scikit-learn.org/stable/common_pitfalls.html), [evaluación](https://scikit-learn.org/stable/modules/model_evaluation.html), [validación temporal](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html).

## Cierre de comprensión
Relaciona cada resultado con su pregunta: ¿qué se predice?, ¿qué datos aprendieron la preparación?, ¿qué significa cada eje?, ¿qué referencia se comparó?, ¿qué error importa? La conversación se realiza con el docente, sin entregables.

Fuentes: [scikit-learn: errores frecuentes](https://scikit-learn.org/stable/common_pitfalls.html), [evaluación](https://scikit-learn.org/stable/modules/model_evaluation.html), [validación temporal](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html).